## 1. Carga y unificación del dataset
Se cargan y concatenan los 112 archivos CSV del hhblock_dataset en un único DataFrame.

In [2]:
import pandas as pd
import glob

archivos = sorted(glob.glob("./data/hhblock_dataset/block_*.csv"))
print(f"Total de archivos encontrados: {len(archivos)}")

lista_df = []
for archivo in archivos:
    df_temp = pd.read_csv(archivo)
    lista_df.append(df_temp)

df = pd.concat(lista_df, ignore_index=True)
print(df.shape)
del lista_df

Total de archivos encontrados: 112
(3469352, 50)


## 2. Análisis inicial de los datos
Se revisa la estructura general del dataset: dimensiones, tipos de datos, valores nulos y duplicados.

In [3]:
print("Filas y columnas:", df.shape)
print("\nTipos de datos:\n", df.dtypes.value_counts())
print("\nNulos por columna (solo las que tienen >0):")
nulos = df.isnull().sum()
print(nulos[nulos > 0])
print("\nTotal de nulos:", df.isnull().sum().sum())
print("\nDuplicados exactos:", df.duplicated().sum())
print("\nDuplicados por LCLid+day (no debería haber):", df.duplicated(subset=["LCLid","day"]).sum())

Filas y columnas: (3469352, 50)

Tipos de datos:
 float64    48
object      2
Name: count, dtype: int64

Nulos por columna (solo las que tienen >0):
hh_19       2
hh_25      21
hh_26       2
hh_30    5460
hh_36       1
dtype: int64

Total de nulos: 5486

Duplicados exactos: 0

Duplicados por LCLid+day (no debería haber): 0


## 3. Tratamiento de valores nulos
Se identificaron 5,486 valores nulos (0.0033% del total), concentrados principalmente en la columna
hh_30, lo cual es consistente con el cambio de horario de verano (DST) del Reino Unido. Se aplica
interpolación horizontal usando las medias horas vecinas del mismo día.

In [7]:
cols_hh = [c for c in df.columns if c.startswith("hh_")]

# Interpolación horizontal: rellena el hueco usando las medias horas vecinas del mismo día
df[cols_hh] = df[cols_hh].interpolate(axis=1, limit_direction="both")

print("Nulos restantes:", df[cols_hh].isnull().sum().sum())

Nulos restantes: 0


## 4. Corrección de tipos de datos
Se convierte `day` a datetime y `LCLid` a category para optimizar memoria y facilitar el análisis temporal.

In [8]:
df["day"] = pd.to_datetime(df["day"])
df["LCLid"] = df["LCLid"].astype("category")

print(df.dtypes.value_counts())
print(df["day"].min(), "a", df["day"].max())

float64           48
category           1
datetime64[ns]     1
Name: count, dtype: int64
2011-11-24 00:00:00 a 2014-02-27 00:00:00


## 5. Dataset limpio final
Se exporta el dataset limpio (3,469,352 filas, 50 columnas, sin nulos ni duplicados) para su uso en
la siguiente etapa de feature engineering y modelado con K-Means (Entregable 2).

In [9]:
df.to_csv("./data/hhblock_dataset_limpio.csv", index=False)
print("Dataset limpio guardado:", df.shape)

Dataset limpio guardado: (3469352, 50)
